In [8]:
# サイネージHTML 生成 (Google Colab / Jupyter)
#
# 概要:
# このスクリプトは、ipywidgetsベースのUIを提供し、
# 写真とテキストメッセージを組み合わせたデジタルサイネージ用の
# スタンドアロンHTMLファイルを生成します。
#
# 2つのモードをサポート:
# 1. Plain (CSS/GSAP): 軽量で安定。写真とテキストを個別にループ。
# 2. Pixi (Particles): 高度な演出。写真やテキストを粒子状にモーフィング。
#
# 生成されるHTMLは、Wake Lock (スリープ防止)、フルスクリーン切り替え、
# CDNフォールバックなどのサイネージ向け最適化を含みます。
#
# ライセンス:
# - このノートブックのコードは MIT License (c) 2025 HosoyaYusaku
# - 同梱/外部ライブラリのライセンスは各プロジェクトに従います（例: GSAP / PixiJS）
#   - GSAP: https://gsap.com/licensing/
#   - PixiJS: https://github.com/pixijs/pixijs/blob/main/LICENSE

from __future__ import annotations

from ipywidgets import (
    FileUpload, Textarea, Dropdown, ColorPicker, FloatSlider, Checkbox, Button,
    VBox, HBox, Layout, ToggleButtons, HTML as WHTML, Select, BoundedIntText, Output, Label
)
from IPython.display import display, HTML
from string import Template
import base64, re, json, io, os, math

# ======= CDN / SRI 設定（固定バージョン）=======
# GSAP 3.12.5（cdnjs SRI 付与）
GSAP_URL = "https://cdnjs.cloudflare.com/ajax/libs/gsap/3.12.5/gsap.min.js"
GSAP_SRI = "sha512-7eHRwcbYkK4d9g/6tD/mhkf++eoTHwpNM9woBxtPUBWm67zeAfFC+HrdoE2GanKeocly/VxeLvIqwvCdk7qScg=="
# PixiJS 7.3.3（SRIは後付け可能。機能に影響しないため現時点では未付与）
PIXI_URL = "https://cdnjs.cloudflare.com/ajax/libs/pixi.js/7.3.3/pixi.min.js"

# 画像軽量化（任意）
try:
    from PIL import Image, ImageOps
    PIL_OK = True
except Exception:
    PIL_OK = False

# Colab DL
try:
    from google.colab import files as gcolab_files
except Exception:
    gcolab_files = None

# ラベルの折返し＋トーストのCSS
display(HTML("""
<style>
.widget-label,label{white-space:normal!important}
.signage-toast{position:fixed;right:16px;bottom:16px;z-index:10000;max-width:420px}
.signage-toast .card{box-shadow:0 10px 30px rgba(0,0,0,.5);border-radius:12px;padding:12px 12px;border:1px solid #f44;background:linear-gradient(#2b0b0b,#180707);color:#fff}
.signage-toast .title{font-weight:700;margin:0 0 4px;font-size:14px}
.signage-toast .desc{font-size:12px;opacity:.9;margin:0 0 8px;line-height:1.5}
.signage-toast .actions > *{margin-right:8px}
.signage-toast .tag{display:inline-block;background:#f33;color:#fff;border-radius:6px;padding:2px 6px;font-size:11px;margin-right:6px}
/* Step見出しのスタイル */
h2.step-header {
    margin-top: 1.5rem;
    margin-bottom: 0.5rem;
    border-bottom: 2px solid #444;
    padding-bottom: 6px;
    font-size: 1.2em;
}
/* UI要素間のマージン調整 */
.widget-box > .widget-label { margin-top: 0.5em; }
</style>
"""))

# ========== utils ==========

def to_data_url(content: bytes, filename: str) -> str:
    name = (filename or "").lower()
    mime = "image/png"
    if name.endswith((".jpg", ".jpeg")):
        mime = "image/jpeg"
    elif name.endswith(".webp"):
        mime = "image/webp"
    elif name.endswith(".gif"):
        mime = "image/gif"
    return f"data:{mime};base64,{base64.b64encode(content).decode('ascii')}"

def optimize_image(content: bytes, filename: str, max_side: int = 1920, quality: int = 90) -> tuple[str, str]:
    """軽量化。Pillowがない時は元のまま。返り値: (data_url, new_name)"""
    if not PIL_OK:
        return to_data_url(content, filename), filename
    try:
        im = Image.open(io.BytesIO(content))
        im = ImageOps.exif_transpose(im)
        im_format = "WEBP"
        # サイズ制限
        w, h = im.size
        if max(w, h) > max_side:
            im.thumbnail((max_side, max_side))
        # RGBA→RGB
        if im.mode in ("RGBA", "P"):
            bg = Image.new("RGB", im.size, (0, 0, 0))
            bg.paste(im, mask=im.split()[-1] if im.mode == "RGBA" else None)
            im = bg
        else:
            im = im.convert("RGB")
        buf = io.BytesIO()
        im.save(buf, format=im_format, quality=quality, method=6)
        data = buf.getvalue()
        new_name = re.sub(r"\.[^.]+$", "", filename or "image") + ".webp"
        return to_data_url(data, new_name), new_name
    except Exception:
        # 失敗時は元のまま
        return to_data_url(content, filename), filename

def parse_lines_to_msgs(raw: str, default_secs: float):
    """1行=1メッセージ。 '秒|テキスト' 形式なら秒を個別指定。"""
    out = []
    for line in (raw or "").splitlines():
        s = line.strip()
        if not s:
            continue
        m = re.match(r"^\s*(\d+(?:\.\d+)?)\s*\|\s*(.+)$", s)
        if m:
            out.append({"text": m.group(2).strip(), "secs": float(m.group(1))})
        else:
            out.append({"text": s, "secs": float(default_secs)})
    return out or [{"text": "サンプル１", "secs": default_secs}]

# JS 文字列安全化（</script> の混入でタグが閉じないように）
def js_str(s: str) -> str:
    return json.dumps(s).replace("</", "<\\/")

# 画面操作レジェンド
LEGEND_HTML = """
<div id="legend" role="status" aria-live="polite" style="position:fixed;left:16px;bottom:16px;z-index:9999;padding:10px 12px;border-radius:10px;background:rgba(20,20,20,.92);border:1px solid #3a3;color:#cfe;font-size:12px;opacity:0;pointer-events:none;transition:opacity .2s">
  <div><b>再生/一時停止</b>：スペース　<b>全画面</b>：F　<b>最初から</b>：R</div>
</div>
<script>
(function(){
  let t=null, el=null; function get(){return el||(el=document.getElementById('legend'))}
  function show(){const b=get(); if(!b)return; b.style.opacity='1'; clearTimeout(t); t=setTimeout(()=>b.style.opacity='0', 2000);}
  ['keydown','mousemove','click','wheel','touchstart'].forEach(ev=>addEventListener(ev,show,{passive:true}));
})();
</script>
"""

# 初回フルスクリーン案内
FSTIP_HTML = """
<div id="fstip" role="note" style="position:fixed;right:16px;top:16px;z-index:9999;padding:10px 12px;border-radius:10px;background:rgba(0,0,0,.8);border:1px solid #6cf;color:#e8f8ff;font-size:12px;opacity:0;pointer-events:none;transition:opacity .5s">
  <div><b>F</b> で全画面 / <b>Space</b> で一時停止</div>
</div>
<script>
(function(){
  try{
    const KEY='signage_tip_seen';
    const el=document.getElementById('fstip');
    if(!el) return;
    if(localStorage.getItem(KEY)==='1'){ el.remove(); return; }
    requestAnimationFrame(()=>{ el.style.opacity='1'; setTimeout(()=>{ el.style.opacity='0'; localStorage.setItem(KEY,'1'); }, 4000); });
  }catch(e){}
})();
</script>
"""

# CDN/非対応フォールバック
CDN_FALLBACK_HTML = """
<div id="cdnfail" style="display:none;position:fixed;inset:0;z-index:10000;display:none;place-items:center;text-align:center;background:rgba(0,0,0,.92);color:#fff;padding:6vw">
  <div style="max-width:1000px">
    <h1 style="margin:.2em 0 0;font-size:clamp(24px,4.5vmin,40px);letter-spacing:.02em">必要なライブラリを読み込めませんでした</h1>
    <p style="font-size:clamp(14px,2.4vmin,20px);opacity:.95;line-height:1.7;margin:.6em 0 1em">
      ネットワーク/プロキシの制限、または一部ブラウザ機能が無効の可能性があります。<br>
      オフライン環境では、CDN の代わりに同梱ファイルを参照する構成をご検討ください。<br>
      <kbd>R</kbd> で再読み込み、<kbd>F</kbd> で全画面の切替ができます。
    </p>
  </div>
</div>
"""

# ========== Plain (CSS/GSAP) ==========

def build_plain_html(slides, messages, bg, tc, ac, fit, transition, pos, size_vmin, loop, img_secs):
    pos_css = {
        "top": "top:5vh;justify-content:flex-start;",
        "center": "top:50%;transform:translateY(-50%);",
        "bottom": "bottom:5vh;justify-content:flex-end;",
    }.get(pos, "bottom:5vh;justify-content:flex-end;")
    messages_array = ",\n        ".join([f"{{ text:{js_str(m['text'])}, secs:{m['secs']} }}" for m in messages])
    slides_array = ",\n        ".join([json.dumps(u) for u in slides])
    slides_markup = "".join([f'<div class="slide" id="slide-{i}"><img src="{u}" /></div>' for i, u in enumerate(slides)])
    empty_first = '<div class="slide active" id="slide-0"></div>' if not slides else ""

    trans_code = f"""
      function showSlide(idx){{
        const prev=document.querySelector('.slide.active');
        const next=document.getElementById('slide-'+idx);
        if(!next) return;
        if(prev && prev!==next && typeof gsap!=='undefined'){{
          prev.classList.remove('active');
          {"gsap.to(prev,{ xPercent:-100, duration:.6, ease:'power2.in' });" if transition=='slide' else "gsap.to(prev,{ opacity:0, duration:.6, ease:'power2.inOut' });"}
        }} else if(prev) {{
          prev.classList.remove('active');
        }}
        next.classList.add('active');
        if(typeof gsap!=='undefined'){{
          {"gsap.fromTo(next,{ xPercent:100 },{ xPercent:0, duration:.6, ease:'power2.out' });" if transition=='slide' else "gsap.fromTo(next,{ opacity:0 },{ opacity:1, duration:.8, ease:'power2.out' });"}
        }}
      }}
      function delay(ms){{return new Promise(r=>setTimeout(r,ms));}}
      let PAUSED=false;
      function setPaused(p){{PAUSED=p; if(typeof gsap!=='undefined') gsap.globalTimeline.paused(p);}}
      async function waitOrPause(ms){{
        const chunk=10;
        let remain=ms;
        while(remain>0){{
          if(PAUSED){{await delay(16); continue;}}
          const d=Math.min(chunk,remain); await delay(d); remain-=d;
        }}
      }}
      function showMessageOnce(msg){{
        return new Promise(async (resolve)=>{{
          const box=document.getElementById('msg');
          box.textContent=msg.text;
          if(typeof gsap!=='undefined'){{
            gsap.killTweensOf(box);
            gsap.fromTo(box,{{autoAlpha:0,y:10}},{{autoAlpha:1,y:0,duration:.35,ease:'power2.out'}});
          }} else {{
            box.style.opacity=1;
          }}
          await waitOrPause(Math.max(300,msg.secs*1000));
          if(typeof gsap!=='undefined'){{
            gsap.to(box,{{autoAlpha:0,y:-6,duration:.3,ease:'power1.inOut',onComplete:resolve}});
          }} else {{
            box.style.opacity=0; resolve();
          }}
        }});
      }}
      async function runSlides(){{
        if(SLIDES.length===0) return;
        SLIDES.forEach(src=>{{const im=new Image(); im.src=src;}});
        let i=0; showSlide(0);
        if(!LOOP && SLIDES.length===1) return;
        while(true){{
          await waitOrPause(IMG_SECS*1000);
          i=(i+1)%SLIDES.length; showSlide(i);
          if(!LOOP && i===SLIDES.length-1) break;
        }}
      }}
      async function runMessages(){{
        if(MESSAGES.length===0) return;
        let k=0;
        while(true){{
          await showMessageOnce(MESSAGES[k%MESSAGES.length]); k++;
          if(!LOOP && k>=MESSAGES.length) break;
        }}
      }}
    """

    tpl = Template(r"""
<!doctype html><html lang="ja"><head>
<meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>Signage (Plain)</title>
<link rel="preconnect" href="https://fonts.googleapis.com" crossorigin>
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Noto+Sans+JP:wght@400;700&display=swap" rel="stylesheet">
<style>
:root{--bg:$bg;--text:$tc;--accent:$ac;--fit:$fit}
html,body{height:100%;margin:0;background:var(--bg);color:var(--text);font-family:'Noto Sans JP',system-ui,-apple-system,'Segoe UI',Roboto,'Hiragino Kaku Gothic ProN',Meiryo,sans-serif}
.stage{position:fixed;inset:0;overflow:hidden;display:grid;place-items:center}
.slide{position:absolute;inset:0;display:grid;place-items:center;opacity:0}
.slide.active{opacity:1}
.slide img{width:100%;height:100%;object-fit:var(--fit)}
.overlay{position:absolute;left:0;right:0;$pos_css display:flex;width:100%;padding:0 5vw;box-sizing:border-box}
#msg{margin:0 auto;font-weight:700;font-size:${size}vmin;line-height:1.2;text-shadow:0 0 10px rgba(0,0,0,.5);letter-spacing:.02em;opacity:0}
.neon{position:absolute;inset:1.2vh 1.2vw;border-radius:24px;box-shadow:0 0 20px var(--accent),0 0 60px rgba(255,255,255,.08) inset;border:2px solid var(--accent);pointer-events:none}
body.no-slides .neon{opacity:.5}
@media (prefers-reduced-motion: reduce){
  .neon{box-shadow:0 0 12px var(--accent),0 0 20px rgba(255,255,255,.04) inset}
}
kbd{background:#111;border:1px solid #444;border-bottom-color:#222;border-radius:4px;padding:.05em .35em}
</style></head>
<body>
  <div class="stage">
    $slides_markup
    $empty_first
    <div class="overlay"><div id="msg"></div></div>
    <div class="neon"></div>
  </div>
  $legend
  $fstip
  $cdn_fallback
  <script src="$gsap_url" integrity="$gsap_sri" crossorigin="anonymous" referrerpolicy="no-referrer"></script>
  <script>
    const SLIDES=[$slides_array];
    const MESSAGES=[$messages_array];
    const LOOP=$loop_bool;
    const IMG_SECS=$img_secs;
    if (SLIDES.length===0) document.body.classList.add('no-slides');

    // prefers-reduced-motion ならややスロウに
    const prefersReduced = (window.matchMedia && window.matchMedia('(prefers-reduced-motion: reduce)').matches);
    function applyReducedMotion(){
      if(prefersReduced && typeof gsap!=='undefined'){ try{ gsap.globalTimeline.timeScale(0.7); }catch(e){} }
    }

    // Wake Lock（画面スリープ防止）
    (function(){
      let lock=null;
      async function keepAwake(){ try{ lock = await navigator.wakeLock?.request?.('screen'); }catch(e){} }
      if('wakeLock' in navigator){ document.addEventListener('visibilitychange',()=>{ if(document.visibilityState==='visible' && !lock) keepAwake(); }); keepAwake(); }
    })();

    $trans_code

    window.addEventListener('load',()=>{
      if(typeof gsap==='undefined'){ const cf=document.getElementById('cdnfail'); if(cf) cf.style.display='grid'; return; }
      applyReducedMotion();
      runSlides(); runMessages();
    },{once:true});

    document.addEventListener('keydown',e=>{
      const tag=(e.target||{}).tagName||'';
      if(tag==='INPUT'||tag==='TEXTAREA') return;
      const k=e.key.toLowerCase();
      if(k==='f'){ const el=document.documentElement; (!document.fullscreenElement? el.requestFullscreen?.():document.exitFullscreen?.()); }
      if(k==='r'){ location.reload(); }
      if(e.code==='Space'){ e.preventDefault(); setPaused(!(typeof gsap!=='undefined' ? gsap.globalTimeline.paused():false)); }
    },{passive:false});
  </script>
</body></html>
""")
    return tpl.substitute(
        bg=bg, tc=tc, ac=ac, fit=fit, pos_css=pos_css, size=size_vmin,
        slides_markup=slides_markup, empty_first=empty_first,
        slides_array=slides_array, messages_array=messages_array,
        loop_bool=str(bool(loop)).lower(), img_secs=img_secs,
        trans_code=trans_code, legend=LEGEND_HTML, fstip=FSTIP_HTML, cdn_fallback=CDN_FALLBACK_HTML,
        gsap_url=GSAP_URL, gsap_sri=GSAP_SRI
    )

# ========== Pixi (Particles) ==========

def build_pixi_html(sequence, bg, tc, ac, cell=7, dot_size=5, morph=2.6, drift=8.0, loop=True, default_img_secs=4.0):
    seq_js = ",\n      ".join([
        ("{ kind:'text', text:%s, secs:%s }" % (js_str(it["text"]), it.get("secs", 3)))
        if it['kind'] == 'text' else
        ("{ kind:'img', url:%s, secs:%s }" % (js_str(it["url"]), it.get("secs", default_img_secs)))
        for it in sequence
    ])

    tpl = Template(r"""
<!doctype html><html lang="ja"><head>
<meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>Signage (Pixi Particles)</title>
<link rel="preconnect" href="https://fonts.googleapis.com" crossorigin>
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Noto+Sans+JP:wght@400;700&display=swap" rel="stylesheet">
<style>
html,body{height:100%;margin:0;background:$bg;color:$tc;font-family:'Noto Sans JP',system-ui,-apple-system,'Segoe UI',Roboto,'Hiragino Kaku Gothic ProN',Meiryo,sans-serif}
canvas{display:block}
#fallback{position:fixed;inset:0;display:none;place-items:center;text-align:center;padding:4vw}
#fallback h2{font-size:clamp(20px,4.2vmin,36px);margin:.2em 0}
#fallback p{font-size:clamp(13px,2.2vmin,18px);line-height:1.7;opacity:.95}
kbd{background:#111;border:1px solid #444;border-bottom-color:#222;border-radius:4px;padding:.05em .35em}
</style>
</head><body>
$legend
$fstip
$cdn_fallback
<div id="fallback"><div>
  <h2>お使いの環境では粒子版が使えません</h2>
  <p>画像や文字は表示できますが、動きは簡易表示になります。<br>必要なら通常版で作り直してください。</p>
</div></div>
<script>
function hasWebGL(){
  try{ const c=document.createElement('canvas'); return !!(c.getContext('webgl')||c.getContext('experimental-webgl')); }catch(e){return false;}
}
</script>
<script src="$pixi_url" crossorigin="anonymous" referrerpolicy="no-referrer"></script>
<script src="$gsap_url" integrity="$gsap_sri" crossorigin="anonymous" referrerpolicy="no-referrer"></script>
<script>
const CFG={ dot:{ cell:$cell, size:$dot_size, alphaThreshold:8, brightThreshold:12 }, morph:$morph, drift:$drift };
const SEQ=[ $seq_js ];
const LOOP=$loop_bool;
let app, layer, dotTexture, tl; let cache={ textPts:{}, imgPts:{} };

function init(){
  if(!hasWebGL()){ document.getElementById('fallback').style.display='grid'; }
  app=new PIXI.Application({ resizeTo:window, background:'$bg', antialias:true });
  document.body.appendChild(app.view);
  layer=new PIXI.Container(); app.stage.addChild(layer);
  const w=app.renderer.width, h=app.renderer.height;
  const border=new PIXI.Container(); app.stage.addChild(border);
  const g1=new PIXI.Graphics(); g1.lineStyle(6, 0x39ff14, 1).drawRoundedRect(6,6,w-12,h-12,48);
  if(PIXI.filters&&PIXI.filters.BlurFilter) g1.filters=[new PIXI.filters.BlurFilter(12)];
  border.addChild(g1);
  const g2=new PIXI.Graphics(); g2.lineStyle(2,0x39ff14,1).drawRoundedRect(6,6,w-12,h-12,48); border.addChild(g2);
  dotTexture=makeDotTexture(CFG.dot.size);
  run();
}
function makeDotTexture(r){ const g=new PIXI.Graphics(); g.beginFill(0xffffff).drawCircle(0,0,r).endFill(); return app.renderer.generateTexture(g); }
function loadImage(src){ return new Promise((res,rej)=>{ const img=new Image(); img.onload=()=>res(img); img.onerror=()=>rej(new Error('画像読み込み失敗')); img.src=src; }); }
function sample(displayObject, targetW){
  const o={ cell:CFG.dot.cell, alphaThreshold:8, brightThreshold:12, targetWidth: targetW||app.renderer.width*0.6 };
  const temp=new PIXI.Container(); const bounds=displayObject.getLocalBounds();
  const scale=o.targetWidth/Math.max(1,bounds.width); displayObject.scale.set(scale); temp.addChild(displayObject);
  const w=Math.ceil(displayObject.width), h=Math.ceil(displayObject.height);
  const rt=PIXI.RenderTexture.create({width:w,height:h,resolution:1}); app.renderer.render(temp,{renderTexture:rt,clear:true});
  const px=app.renderer.extract.pixels(rt), pts=[];
  for(let y=0;y<h;y+=o.cell){ for(let x=0;x<w;x+=o.cell){
    const i=(y*w+x)*4; const r=px[i],g=px[i+1],b=px[i+2],a=px[i+3];
    const bright=(r+g+b)/3; if(a>o.alphaThreshold||bright>o.brightThreshold) pts.push({x,y,color:(r<<16)|(g<<8)|b});
  }}
  const ox=(app.renderer.width-w)/2, oy=(app.renderer.height-h)/2; pts.forEach(p=>{p.x+=ox; p.y+=oy;});
  rt.destroy(true); temp.destroy({children:true}); return pts;
}
function textPoints(t){
  if(cache.textPts[t]) return cache.textPts[t];
  const style=new PIXI.TextStyle({ fill:'#ffffff', fontSize:200, fontWeight:'700', letterSpacing:2 });
  const txt=new PIXI.Text(t, style); const pts=sample(txt, app.renderer.width*0.7);
  pts.forEach(p=>p.color=0xeffff4); cache.textPts[t]=pts; return pts;
}
async function imgPoints(url){
  if(cache.imgPts[url]) return cache.imgPts[url];
  const img=await loadImage(url); const tex=PIXI.Texture.from(img); const sp=new PIXI.Sprite(tex);
  const pts=sample(sp, app.renderer.width*0.6); cache.imgPts[url]=pts; return pts;
}
function makeDots(n){
  const sprites=[]; const W=app.renderer.width, H=app.renderer.height;
  for(let i=0;i<n;i++){ const s=new PIXI.Sprite(dotTexture); s.anchor.set(0.5);
    const side=Math.floor(Math.random()*4);
    if(side===0){s.x=Math.random()*W; s.y=-20;} else if(side===1){s.x=W+20; s.y=Math.random()*H;}
    else if(side===2){s.x=Math.random()*W; s.y=H+20;} else {s.x=-20; s.y=Math.random()*H;}
    s.alpha=0; s.tint=0x39ff14; layer.addChild(s);
    if(typeof gsap!=='undefined') gsap.to(s,{x:'+='+(Math.random()*10-5), y:'+='+(Math.random()*10-5), duration:CFG.drift, ease:'sine.inOut', repeat:-1, yoyo:true});
    sprites.push(s);
  } return sprites;
}
function morphTo(points, duration){
  if(!points.length) return;
  const sprites=layer.children;
  const xs=points.map(p=>p.x); const minX=Math.min(...xs), maxX=Math.max(...xs);
  const span=Math.max(1,maxX-minX), STAGGER=.28;
  for(let i=0;i<sprites.length;i++){
    const s=sprites[i]; const p=points[i%points.length];
    const delay=((p.x-minX)/span)*STAGGER + (i%7)*0.008;
    s.tint=p.color||s.tint;
    if(typeof gsap!=='undefined') gsap.to(s,{x:p.x,y:p.y,alpha:1,duration:duration,delay:delay,ease:'power2.out'});
    else { s.x=p.x; s.y=p.y; s.alpha=1; }
  }
}
async function run(){
  if(tl){ tl.kill(); tl=null; } layer.removeChildren();
  let base=1000; for(const it of SEQ){ if(it.kind==='text'){ base=Math.max(base,textPoints(it.text).length); } else { base=Math.max(base,(await imgPoints(it.url)).length); } }
  makeDots(base); tl=(typeof gsap!=='undefined')? gsap.timeline({ repeat: LOOP?-1:0, repeatDelay:.6 }): null;
  for(const it of SEQ){
    if(it.kind==='text'){ const pts=textPoints(it.text); if(tl){ tl.call(()=>morphTo(pts, CFG.morph)); tl.to({}, {duration:Math.max(.6,it.secs||3)}); } else { morphTo(pts, CFG.morph); } }
    else { const pts=await imgPoints(it.url); if(tl){ tl.call(()=>morphTo(pts, CFG.morph)); tl.to({}, {duration:Math.max(.6,it.secs||4)}); } else { morphTo(pts, CFG.morph); } }
  }
  document.addEventListener('keydown',(e)=>{
    const tag=(e.target||{}).tagName||''; if(tag==='INPUT'||tag==='TEXTAREA') return;
    const k=(e.key||'').toLowerCase();
    if(k==='f'){ const el=document.documentElement; (!document.fullscreenElement? el.requestFullscreen?.():document.exitFullscreen?.()); }
    if(k==='r'){ location.reload(); }
    if(e.code==='Space'){ e.preventDefault(); const p=!((typeof gsap!=='undefined')?gsap.globalTimeline.paused():false); if(tl) tl.paused(p); if(typeof gsap!=='undefined') gsap.globalTimeline.paused(p); }
  },{passive:false});
}

// prefers-reduced-motion → タイムスケール
(function(){
  const prefersReduced = (window.matchMedia && window.matchMedia('(prefers-reduced-motion: reduce)').matches);
  if(prefersReduced && typeof gsap!=='undefined'){ try{ gsap.globalTimeline.timeScale(0.7); }catch(e){} }
})();

// Wake Lock
(function(){
  let lock=null;
  async function keepAwake(){ try{ lock = await navigator.wakeLock?.request?.('screen'); }catch(e){} }
  if('wakeLock' in navigator){ document.addEventListener('visibilitychange',()=>{ if(document.visibilityState==='visible' && !lock) keepAwake(); }); keepAwake(); }
})();

// 起動
if(document.readyState==='complete'){
  if(typeof PIXI==='undefined' || typeof gsap==='undefined'){ const cf=document.getElementById('cdnfail'); if(cf) cf.style.display='grid'; }
  else{ init(); }
}else{
  addEventListener('load', ()=>{
    if(typeof PIXI==='undefined' || typeof gsap==='undefined'){ const cf=document.getElementById('cdnfail'); if(cf) cf.style.display='grid'; }
    else{ init(); }
  }, {once:true});
}
</script></body></html>
""")
    return tpl.substitute(
        bg=bg, tc=tc, ac=ac,
        cell=int(cell), dot_size=int(dot_size),
        morph=float(morph), drift=float(drift),
        loop_bool=str(bool(loop)).lower(),
        seq_js=seq_js, legend=LEGEND_HTML, fstip=FSTIP_HTML, cdn_fallback=CDN_FALLBACK_HTML,
        pixi_url=PIXI_URL, gsap_url=GSAP_URL, gsap_sri=GSAP_SRI
    )

# ========== UI ==========

H = lambda s: WHTML(value=f"<h4 style='margin:1rem 0 .2rem'>{s}</h4>")
P = lambda s: WHTML(value=f"<div style='font-size:12px;opacity:.8;margin:-.2rem 0 .4rem;line-height:1.5;'>{s}</div>")
HStep = lambda s: WHTML(value=f"<h2 class='step-header'>{s}</h2>")
log = Output()

# トースト
toast_box = VBox([], layout=Layout(width="420px"))
toast_box.add_class("signage-toast")

def hide_toast(_=None):
    toast_box.children = ()

def show_error_toast(title: str, details: list[str], suggest_text=True, suggest_image=True):
    ttl = WHTML(f"<div class='title'><span class='tag'>注意</span>{title}</div>")
    desc_html = "<ul style='margin:.2em 0 .4em;padding-left:1.2em'>"+ "".join([f"<li>{d}</li>" for d in details]) + "</ul>"
    desc = WHTML(f"<div class='desc'>{desc_html}</div>")

    actions_widgets = []
    if suggest_text:
        btn1 = Button(description="✍️ 文言を追加（現在の入力を追加）", button_style="warning", layout=Layout(width="100%"))
        def _add_text(_):
            add_text_items()
            hide_toast()
        btn1.on_click(_add_text)
        actions_widgets.append(btn1)
    if suggest_image:
        btn2 = Button(description="🖼 写真をアップロード", layout=Layout(width="100%"))
        def _hint_img(_):
            with log:
                print("上の『Step 1a: 写真のアップロード』で画像を選択してください。追加後、順序エディタで位置を調整できます。")
            hide_toast()
        btn2.on_click(_hint_img)
        actions_widgets.append(btn2)
    btn_close = Button(description="閉じる", layout=Layout(width="100%"))
    btn_close.on_click(hide_toast)
    actions = VBox(actions_widgets+[btn_close], layout=Layout(width="100%"), _dom_classes=["actions"])
    card = VBox([ttl, desc, actions], _dom_classes=["card"])
    toast_box.children = (card,)

# --- Step 1: コンテンツ準備 ---
uploader = FileUpload(description="写真を選択", accept='image/*', multiple=True, layout=Layout(width="100%"))
image_lib: dict[str, str] = {}
processed_original_names: set[str] = set()
is_upload_processing: bool = False
lighten = Checkbox(value=True, description="軽量化して追加（推奨）")
max_side = BoundedIntText(value=1920, min=640, max=4096, step=10, layout=Layout(width="120px"))

# ← ここを「サンプル１」のみに変更
text = Textarea(
    value="サンプル１",
    layout=Layout(width="100%", height="120px"),
)
txt_secs_default = FloatSlider(description="文字の既定秒数（秒指定がない行に適用）", min=1, max=15, step=0.5, value=4, layout=Layout(width="100%"), style={'description_width': 'initial'})

def on_uploaded(change=None):
    global is_upload_processing, processed_original_names

    if is_upload_processing:
        return
    if not uploader.value:
        return

    try:
        is_upload_processing = True
        files_to_process = uploader.value.copy()

        new_files_count = 0
        with log:
            log.clear_output(wait=True)
            print(f"{len(files_to_process)} 件のファイルアップロードを検知しました。処理を開始します...")

            for name, meta in files_to_process.items():
                if name in processed_original_names:
                    print(f"  (スキップ: '{name}' は処理済み)")
                    continue

                try:
                    content = meta['content']
                    if lighten.value:
                        url, new_name = optimize_image(content, name, max_side.value)
                    else:
                        url, new_name = to_data_url(content, name), name

                    base, ext = os.path.splitext(new_name)
                    candidate = new_name
                    n = 1
                    while candidate in image_lib:
                        candidate = f"{base}_{n}{ext or '.webp'}"
                        n += 1
                    new_name = candidate

                    image_lib[new_name] = url
                    processed_original_names.add(name)
                    new_files_count += 1

                except Exception as e:
                    print(f"❌ ファイル '{name}' の処理中にエラーが発生しました: {e}")

            img_pick.options = [(n, n) for n in sorted(image_lib.keys())]
            if img_pick.options and not img_pick.value:
                img_pick.value = img_pick.options[0][1]

            if new_files_count > 0:
                print(f"--- 完了: {new_files_count} 件の画像を追加しました。合計 {len(image_lib)} 件 ---")
            else:
                print("--- 完了: 新しく追加された画像はありませんでした ---")

    finally:
        is_upload_processing = False

uploader.observe(on_uploaded, names='value')

# --- Step 2: 順序編集 ---
seq_list = Select(options=[], rows=8, layout=Layout(width="100%", min_width="300px"))
btn_up = Button(description="▲ 上へ", layout=Layout(width="90px"))
btn_down = Button(description="▼ 下へ", layout=Layout(width="90px"))
btn_del = Button(description="Ⓧ 削除", layout=Layout(width="90px"))

btn_add_text = Button(description="＋ 入力した文言を末尾に追加", button_style="info", layout=Layout(width="100%"))
img_pick = Dropdown(description="写真を選択:", options=[], layout=Layout(width="100%", min_width="200px"), style={'description_width': 'initial'})
insert_pos = BoundedIntText(description="挿入位置(行):", value=0, min=0, max=0, step=1, layout=Layout(width="180px"), style={'description_width': 'initial'})
btn_add_img = Button(description="＋ 写真を挿入", button_style="info", layout=Layout(width="100%"))

sequence: list[dict] = []

def rebuild_seq():
    opts = []
    for i, it in enumerate(sequence):
        if it["kind"] == "text":
            label = f"{i+1:02d}. ✍️ {it['text'][:30]}{'…' if len(it['text'])>30 else ''} ({it.get('secs',4)}s)"
        else:
            label = f"{i+1:02d}. 🖼 {it['name'][:30]}{'…' if len(it['name'])>30 else ''} ({it.get('secs',4)}s)"
        opts.append((label, i))

    current_val = seq_list.value
    seq_list.options = opts
    if current_val in [o[1] for o in opts]:
        seq_list.value = current_val
    elif opts:
        seq_list.value = opts[-1][1]
    else:
        seq_list.value = None

    insert_pos.max = max(0, len(sequence))
    insert_pos.value = len(sequence)

def add_text_items(_=None):
    msgs = parse_lines_to_msgs(text.value, txt_secs_default.value)
    # サンプル１だけは、初回（sequence が空）なら追加を許可
    if not msgs or (len(msgs) == 1 and msgs[0]['text'] == 'サンプル１' and len(sequence) == 0):
         if len(sequence) == 0 and msgs[0]['text'] == 'サンプル１':
             pass
         else:
             with log:
                log.clear_output(wait=True)
                print("追加する文言が入力されていません（サンプル文言は追加されません）。「1b. 表示する文言」に入力してください。")
             return

    added_count = 0
    for m in msgs:
        # 2回目以降は「サンプル１」を無視
        if m['text'] in ["サンプル１"] and len(sequence) > 0:
            continue
        s = max(0.5, min(60.0, float(m["secs"])))
        sequence.append({"kind": "text", "text": m["text"], "secs": s})
        added_count += 1

    rebuild_seq()
    with log:
        log.clear_output(wait=True)
        print(f"{added_count} 件の文言をリストの末尾に追加しました。")

def add_img_item(_=None):
    if not img_pick.options:
        with log:
            log.clear_output(wait=True)
            print("写真がアップロードされていません。「Step 1a」からアップロードしてください。")
        return
    name = img_pick.value
    url = image_lib.get(name)
    if not url:
        with log:
            log.clear_output(wait=True)
            print(f"エラー: 画像 '{name}' が見つかりません。")
        return

    pos = int(insert_pos.value)
    sequence.insert(pos, {"kind": "img", "name": name, "url": url, "secs": img_secs_pixi.value})
    rebuild_seq()
    with log:
        log.clear_output(wait=True)
        print(f"写真 '{name}' を {pos+1} 行目に挿入しました。")

def move_up(_):
    if seq_list.value is None: return
    i = int(seq_list.value)
    if i <= 0: return
    sequence[i - 1], sequence[i] = sequence[i], sequence[i - 1]
    rebuild_seq()
    seq_list.value = i - 1

def move_down(_):
    if seq_list.value is None: return
    i = int(seq_list.value)
    if i >= len(sequence) - 1: return
    sequence[i + 1], sequence[i] = sequence[i], sequence[i + 1]
    rebuild_seq()
    seq_list.value = i + 1

def delete_item(_):
    if seq_list.value is None: return
    deleted_item = sequence.pop(int(seq_list.value))
    rebuild_seq()
    with log:
        log.clear_output(wait=True)
        print(f"項目「{deleted_item.get('name') or deleted_item.get('text')}」を削除しました。")

btn_up.on_click(move_up)
btn_down.on_click(move_down)
btn_del.on_click(delete_item)
btn_add_text.on_click(add_text_items)
btn_add_img.on_click(add_img_item)

# --- Step 3: モードとスタイル ---

mode = ToggleButtons(
    options=[("通常（シンプル・軽量）", "plain"), ("Pixi粒子版（高度な演出）", "pixi")],
    value="pixi",
    tooltips=["写真とテキストを個別にループ再生", "Step 2の順序どおりに再生"],
    layout=Layout(width="100%")
)

bg = ColorPicker(description="背景色", value="#000000", layout=Layout(width="220px"))
tc = ColorPicker(description="文字色", value="#EFFFF4", layout=Layout(width="220px"))
ac = ColorPicker(description="アクセント色（枠・発光など）", value="#39FF14", layout=Layout(width="260px"), style={'description_width': 'initial'})
loop = Checkbox(value=True, description="最後まで再生後に繰り返す（ループ）")

fit = Dropdown(
    description="写真の表示方法:",
    options=[("cover（画面いっぱい・はみ出し可）", "cover"), ("contain（全体表示・余白あり）", "contain")],
    value="cover",
    layout=Layout(width="100%"), style={'description_width': 'initial'}
)
transition = Dropdown(
    description="写真の切替演出:",
    options=[("fade（上品な切替）", "fade"), ("slide（動きが明確）", "slide")],
    value="fade",
    layout=Layout(width="100%"), style={'description_width': 'initial'}
)
pos = Dropdown(
    description="テキストの縦位置:",
    options=[("top（上寄せ）", "top"), ("center（中央）", "center"), ("bottom（下寄せ/推奨）", "bottom")],
    value="center",
    layout=Layout(width="100%"), style={'description_width': 'initial'}
)
size = FloatSlider(description="文字サイズ（vmin：画面の短辺が基準）", min=3, max=12, step=0.5, value=7.0, layout=Layout(width="100%"), style={'description_width': 'initial'})
img_secs_plain = FloatSlider(description="写真の表示秒数（全写真共通）", min=1, max=20, step=0.5, value=4, layout=Layout(width="100%"), style={'description_width': 'initial'})

cell = FloatSlider(description="粒子の間隔（小さい=密・重い）", min=5, max=12, step=1, value=7, layout=Layout(width="100%"), style={'description_width': 'initial'})
dot_size = FloatSlider(description="粒子のサイズ（大きい=太め）", min=3, max=8, step=1, value=5, layout=Layout(width="100%"), style={'description_width': 'initial'})
morph = FloatSlider(description="形の切替にかかる秒数（ロゴ⇄文言）", min=0.8, max=6, step=0.2, value=2.6, layout=Layout(width="100%"), style={'description_width': 'initial'})
drift = FloatSlider(description="粒子の揺らぎ周期（秒数）", min=3, max=16, step=0.5, value=8.0, layout=Layout(width="100%"), style={'description_width': 'initial'})
img_secs_pixi = FloatSlider(description="写真の表示秒数（Pixi用・全写真共通）", min=1, max=20, step=0.5, value=4, layout=Layout(width="100%"), style={'description_width': 'initial'})

# --- Step 4: 生成 ---
btn_generate = Button(description="HTMLを生成してダウンロード", button_style="success", layout=Layout(width="100%", height="50px"))

MAX_TEXT_ITEMS = 50
MAX_TEXT_LEN = 120

def validate_sequence():
    """問題があれば（title, [detail,...], suggest_text, suggest_image）を返す。なければ None"""
    if not sequence:
        return ("順序リストが空です", ["「Step 2」で表示したい文言や写真を追加してください。"], True, True)

    img_count = sum(1 for it in sequence if it.get("kind")=="img")
    text_items = [it for it in sequence if it.get("kind")=="text"]

    problems = []
    suggest_text = False
    suggest_image = False

    if img_count == 0:
        problems.append("写真が 0 件です。写真なしでも動作しますが、見栄えを良くするため「Step 1a」でのアップロードを推奨します。")
        suggest_image = True

    if len(text_items) > MAX_TEXT_ITEMS:
        problems.append(f"テキスト項目が多すぎます（{len(text_items)}行）。推奨は {MAX_TEXT_ITEMS} 行以下です。")
        suggest_text = True

    long_lines = [len(x.get("text","")) for x in text_items if len(x.get("text","")) > MAX_TEXT_LEN]
    if long_lines:
        problems.append(f"長すぎる行があります（最大 {max(long_lines)} 文字）。1行 {MAX_TEXT_LEN} 文字以下を推奨します。")
        suggest_text = True

    if problems:
        return ("設定内容の確認", problems, suggest_text, suggest_image)

    return None

def on_generate(_):
    with log:
        log.clear_output()
        print("HTMLを生成中です...")

    check = validate_sequence()
    if check:
        title, details, sug_t, sug_i = check
        show_error_toast(title, details, suggest_text=sug_t, suggest_image=sug_i)
        with log:
            log.clear_output()
            print(f"⚠️ {title}（詳細は右下の通知を確認してください）")
        return

    slides = [it["url"] for it in sequence if it["kind"] == "img"]
    messages = [
        {"text": it["text"], "secs": max(0.5, min(60.0, it.get("secs", 4)))}
        for it in sequence if it["kind"] == "text"
    ]

    img_secs_to_use = img_secs_plain.value if mode.value == "plain" else img_secs_pixi.value

    pixi_sequence = []
    for it in sequence:
        new_it = it.copy()
        if it['kind'] == 'img':
            new_it['secs'] = img_secs_to_use
        else:
            new_it['secs'] = max(0.5, min(60.0, it.get('secs', txt_secs_default.value)))
        pixi_sequence.append(new_it)

    if mode.value == "plain":
        html_str = build_plain_html(
            slides, messages, bg.value, tc.value, ac.value, fit.value, transition.value,
            pos.value, size.value, bool(True if loop.value else False), img_secs_to_use
        )
        filename = "signage_plain.html"
    else:
        html_str = build_pixi_html(
            pixi_sequence, bg.value, tc.value, ac.value, cell.value, dot_size.value,
            morph.value, drift.value, bool(True if loop.value else False), img_secs_to_use
        )
        filename = "signage_pixi.html"

    path = f"/content/{filename}"
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(html_str)

        if gcolab_files:
            with log:
                print(f"✅ {filename} を生成しました。ダウンロードを開始します。")
            gcolab_files.download(path)
        else:
            with log:
                print(f"✅ {filename} を {path} に保存しました。(Colab環境外のため自動ダウンロードなし)")

    except Exception as e:
        with log:
            print(f"❌ ファイルの保存またはダウンロードに失敗しました: {e}")

btn_generate.on_click(on_generate)

# 初期：サンプル文言（サンプル１のみ）
add_text_items()

# ========== レイアウト ==========

step1_upload = VBox([
    H("1a. 写真のアップロード"),
    P("サイネージで表示したい写真をアップロードします。ボタンの (n) には選択したファイル数が表示されます。<br>ここでアップロードした写真は、Step 2 の「写真を追加」プルダウンから選択できます。"),
    uploader,
    HBox([lighten, HBox([Label(value="長辺上限(px):"), max_side])]),
])

step1_text = VBox([
    H("1b. 表示する文言"),
    P("表示したいメッセージを1行に1つずつ入力します。<br>"
      "<b>例:</b> <code>5|セール中</code> と入力すると、その行だけ5秒間表示されます（秒指定がない行は下の既定秒数が適用されます）。<br>"
      "ここで入力した文言は、Step 2 の「文言を末尾に追加」ボタンで順序リストに追加します。"),
    text,
    txt_secs_default,
])

step2_add = VBox([
    H("リストへの追加"),
    P("<b>文言を追加:</b> 上の「1b. 表示する文言」に入力したテキスト（サンプル文言を除く）を、まとめてリストの末尾に追加します。"),
    btn_add_text,
    P("<b>写真を追加:</b> アップロード済みの写真（1a）を選び、挿入したい位置（行番号）を指定してリストに追加します。（0=先頭, 1=2行目, ...）"),
    VBox([img_pick, HBox([insert_pos, btn_add_img])], layout=Layout(width="100%")),
], layout=Layout(width="100%", margin="0 0 1rem 0"))

step2_edit = VBox([
    H("順序リストの操作"),
    P("リスト内の項目を選択し、「▲ 上へ」「▼ 下へ」で順序を入れ替えたり、「Ⓧ 削除」でリストから削除できます。"),
    HBox([seq_list, VBox([btn_up, btn_down, btn_del], layout=Layout(margin="0 0 0 0.5rem"))]),
], layout=Layout(width="100%"))

step3_mode = VBox([
    H("3a. 表示モードの選択"),
    P("「通常」は写真とテキストを個別にループ再生します。「Pixi粒子版」は Step 2 で決めた順序どおりに再生されます。"),
    mode,
])

step3_common = VBox([
    H("3b. 共通スタイル設定"),
    P("どちらのモードを選んでも適用される、基本的な色の設定です。暗所での反射を抑えた黒基調を推奨します。"),
    HBox([bg, tc, ac], layout=Layout(flex_flow="row wrap")),
    loop,
])

step3_plain = VBox([
    H("3c. [通常モード] の詳細設定"),
    P("「通常モード」選択時のみ適用されます。写真とテキストをシンプルに切り替える、軽量で安定したモードです。"),
    fit,
    transition,
    pos,
    size,
    img_secs_plain,
])

step3_pixi = VBox([
    H("3d. [粒子モード] の詳細設定"),
    P("「Pixi粒子版」選択時のみ適用されます。写真や文字を粒子（ドット）で表現し、滑らかに変形させます。"),
    cell,
    dot_size,
    morph,
    drift,
    img_secs_pixi,
])

step4_generate = VBox([
    WHTML(value="<hr style='margin:1rem 0;'>"),
    btn_generate,
    log
])

controls = VBox([
    toast_box,
    WHTML(value="<h3>サイネージHTML 生成（ダウンロード専用）</h3>"
                "<div style='font-size:12px;opacity:.8;margin-bottom:1rem;line-height:1.5;'>"
                "Step 1で素材（写真・文言）を準備し、Step 2で表示順序を決めます。<br>"
                "Step 3で見た目（モードや色）を調整し、Step 4でHTMLファイルを生成・ダウンロードします。"
                "</div>"),
    HStep("Step 1: コンテンツの準備"),
    step1_upload,
    step1_text,
    HStep("Step 2: 表示順序の編集"),
    step2_add,
    step2_edit,
    HStep("Step 3: 表示モードとスタイルの設定"),
    step3_mode,
    step3_common,
    step3_plain,
    step3_pixi,
    HStep("Step 4: 生成とダウンロード"),
    P("すべての設定が完了したら、下のボタンを押して <code>signage_xxx.html</code> ファイルを生成し、ダウンロードします。"),
    step4_generate
])

display(controls)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>